# Phase 3 — Task 3.2: Convex Portfolio Optimizer

**Objective:** Translate alpha signals (μ) and the PCA risk model (Σ) into optimal market-neutral portfolio weights via convex optimization with turnover penalty and layered constraints.

---

**Inputs:**
| File | Description |
|---|---|
| `sp500_expected_returns.csv.gz` | μ(i,t) from Task 2.4 |
| `sp500_risk_model.pkl` | β, Ω\_f, Λ\_ε from Task 3.1 |
| `sp500_monthly_returns.csv.gz` | Realized returns (for drift & NAV) |
| `F-F_Research_Data_5_Factors_2x3_daily.CSV` | FF5 factors (for RF) |

**Outputs:**
| File | Description |
|---|---|
| `sp500_portfolio_weights.csv.gz` | Monthly optimal weights ω\*(i,t) |
| `sp500_backtest_results.csv.gz` | NAV, returns, turnover, leverage |

In [1]:
# ============================================================
# Cell 1: Imports & Load Data
# ============================================================
import pandas as pd
import numpy as np
import cvxpy as cp
import pickle
import time
import os
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "data/"

# 1) Expected returns (alpha signal)
er = pd.read_csv(DATA_DIR + "sp500_expected_returns.csv.gz", compression="gzip")
er["date"] = pd.to_datetime(er["date"])
er["month"] = er["date"].dt.to_period("M")
print(f"Expected returns: {er.shape[0]:,} rows, {er['month'].nunique()} months")

# 2) Risk model
with open(DATA_DIR + "sp500_risk_model.pkl", "rb") as f:
    risk_models = pickle.load(f)
print(f"Risk models: {len(risk_models)} months")

# 3) Monthly returns (for drift weights & NAV)
monthly_ret = pd.read_csv(DATA_DIR + "sp500_monthly_returns.csv.gz",
                          compression="gzip")
monthly_ret["last_date"] = pd.to_datetime(monthly_ret["last_date"])
monthly_ret["month"] = monthly_ret["last_date"].dt.to_period("M")
print(f"Monthly returns: {monthly_ret.shape[0]:,} rows")

# Build permno -> month -> return lookup
ret_lookup = monthly_ret.set_index(["permno", "month"])["ret_monthly"].to_dict()

# 4) Fama-French 5 factors (for risk-free rate)
ff5_raw = pd.read_csv(DATA_DIR + "F-F_Research_Data_5_Factors_2x3_daily.CSV",
                       skiprows=3)
ff5_raw.columns = [c.strip() for c in ff5_raw.columns]
date_col = ff5_raw.columns[0]
ff5_raw = ff5_raw.rename(columns={date_col: "date_raw"})
ff5_raw = ff5_raw[ff5_raw["date_raw"].astype(str).str.match(r"^\d{8}$", na=False)].copy()
ff5_raw["date"] = pd.to_datetime(ff5_raw["date_raw"].astype(str))
for c in ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]:
    ff5_raw[c] = pd.to_numeric(ff5_raw[c], errors="coerce")
ff5_raw["month"] = ff5_raw["date"].dt.to_period("M")

# Monthly RF
rf_monthly = ff5_raw.groupby("month").agg(
    rf_monthly=("RF", lambda x: (1 + x/100).prod() - 1)
).reset_index()
rf_dict = dict(zip(rf_monthly["month"], rf_monthly["rf_monthly"]))
print(f"Monthly RF: {len(rf_dict)} months")

# Monthly FF5 factor returns
ff5_monthly = ff5_raw.groupby("month").agg(
    mktrf=("Mkt-RF", lambda x: (1 + x/100).prod() - 1),
    smb=("SMB", lambda x: (1 + x/100).prod() - 1),
    hml=("HML", lambda x: (1 + x/100).prod() - 1),
    rmw=("RMW", lambda x: (1 + x/100).prod() - 1),
    cma=("CMA", lambda x: (1 + x/100).prod() - 1),
    rf=("RF", lambda x: (1 + x/100).prod() - 1),
).reset_index()
print(f"FF5 monthly: {len(ff5_monthly)} months")

Expected returns: 99,363 rows, 157 months
Risk models: 157 months
Monthly returns: 140,070 rows
Monthly RF: 738 months
FF5 monthly: 738 months


## Step 1: Data Alignment

For each month, take the **intersection** of stocks available in both the alpha signal (μ) and the risk model (Σ), ensuring identical ordering.

In [2]:
# ============================================================
# Cell 2: Data Alignment Function
# ============================================================

def align_month_data(month, er_df, risk_models, ret_lookup):
    '''
    Align alpha signals, risk model, and sector info for one month.

    Returns dict with aligned arrays, or None if insufficient data.
    '''
    month_str = str(month)

    # Alpha signal permnos
    er_m = er_df[er_df["month"] == month]
    if len(er_m) == 0:
        return None
    alpha_permnos = set(er_m["permno"].values)

    # Risk model permnos
    if month_str not in risk_models:
        return None
    rm = risk_models[month_str]
    model_permnos = set(rm["permnos"])

    # Intersection
    common = sorted(alpha_permnos & model_permnos)
    if len(common) < 50:
        return None

    # Alpha vector (aligned to common order)
    er_indexed = er_m.set_index("permno")
    mu = np.array([er_indexed.loc[p, "mu_compound"] for p in common])

    # Sector mapping
    sectors = np.array([int(er_indexed.loc[p, "gsector"]) for p in common])

    # Risk model components (aligned to common order)
    model_idx = {p: i for i, p in enumerate(rm["permnos"])}
    idx_map = [model_idx[p] for p in common]
    beta = rm["beta"][idx_map]
    lambda_eps = rm["lambda_eps"][idx_map]
    omega_f = rm["omega_f"]

    # Realized returns for this month (for next-month NAV calc)
    # We need NEXT month's returns; caller handles that
    next_month = month + 1
    rets = np.array([ret_lookup.get((p, next_month), np.nan) for p in common])

    return {
        "permnos":    common,
        "mu":         mu,
        "beta":       beta,
        "omega_f":    omega_f,
        "lambda_eps": lambda_eps,
        "sectors":    sectors,
        "fwd_ret":    rets,
        "n_stocks":   len(common),
    }


# --- Test alignment on first month ---
rebal_months = sorted(er["month"].unique())
test = align_month_data(rebal_months[0], er, risk_models, ret_lookup)
if test:
    print(f"Alignment test ({rebal_months[0]}):")
    print(f"  Stocks: {test['n_stocks']}")
    print(f"  mu shape: {test['mu'].shape}")
    print(f"  beta shape: {test['beta'].shape}")
    print(f"  omega_f shape: {test['omega_f'].shape}")
    print(f"  lambda_eps shape: {test['lambda_eps'].shape}")
    print(f"  Sectors: {len(np.unique(test['sectors']))}")
    print(f"  Forward returns available: {np.isfinite(test['fwd_ret']).sum()}")

Alignment test (2011-12):
  Stocks: 678
  mu shape: (678,)
  beta shape: (678, 29)
  omega_f shape: (29, 29)
  lambda_eps shape: (678,)
  Sectors: 11
  Forward returns available: 678


## Step 2: cvxpy Optimizer

**Objective (minimize):**
$$-w^T \mu + \frac{\gamma}{2} w^T \Sigma w + c \|w - w_{\text{prev}}\|_1$$

**Efficient variance computation** via factor decomposition:
$$w^T \Sigma w = z^T \Omega_f z + \sum_i \lambda_i w_i^2, \quad z = \beta^T w$$

**Constraint layers:**
- **Layer 1 (always):** dollar neutrality, gross leverage ≤ L\_max, |w\_i| ≤ w\_max
- **Layer 2 (default on):** volatility target, sector neutrality

In [3]:
# ============================================================
# Cell 3: Optimizer Configuration & Core Function
# ============================================================

# --- Default parameters ---
GAMMA         = 1.0      # risk aversion
COST_PENALTY  = 0.005    # turnover L1 penalty (~50bp one-way)
L_MAX         = 2.0      # gross leverage ceiling
W_MAX         = 0.05     # max single-stock |weight|
VOL_TARGET    = 0.10     # annualized vol target
SECTOR_TOL    = 0.02     # max net sector exposure (absolute)

# Monthly vol target
MONTHLY_VOL_SQ = (VOL_TARGET / np.sqrt(12)) ** 2

print("Optimizer parameters:")
print(f"  gamma (risk aversion):    {GAMMA}")
print(f"  cost penalty (c):         {COST_PENALTY}")
print(f"  gross leverage (L_max):   {L_MAX}")
print(f"  single stock (w_max):     {W_MAX}")
print(f"  vol target (ann):         {VOL_TARGET:.0%}")
print(f"  monthly var target:       {MONTHLY_VOL_SQ:.6f}")
print(f"  sector tolerance:         {SECTOR_TOL}")


def build_and_solve(mu, beta, omega_f, lambda_eps, sectors,
                    w_prev, constraint_level=2,
                    sector_tol=SECTOR_TOL, vol_sq=MONTHLY_VOL_SQ,
                    l_max=L_MAX):
    '''
    Build and solve the portfolio optimization problem.

    constraint_level:
        1 = dollar neutral + leverage + stock limits only
        2 = level 1 + volatility target + sector neutrality

    Returns: w_opt (N,), status (str), obj_val (float)
    '''
    N = len(mu)
    K = omega_f.shape[0]

    w = cp.Variable(N)

    # Factor exposure vector z = beta.T @ w  (K x 1)
    z = beta.T @ w

    # Portfolio variance via factor decomposition
    # w'Sigma w = z' Omega z + w' diag(lambda) w
    port_var = cp.quad_form(z, omega_f) + lambda_eps @ cp.square(w)

    # Objective: minimize  -mu'w + (gamma/2)*var + c*||w - w_prev||_1
    objective = cp.Minimize(
        -mu @ w
        + (GAMMA / 2.0) * port_var
        + COST_PENALTY * cp.norm1(w - w_prev)
    )

    # --- Layer 1 constraints (always active) ---
    constraints = [
        cp.sum(w) == 0,                       # dollar neutral
        cp.norm1(w) <= l_max,                  # gross leverage
        w >= -W_MAX,                           # individual lower bound
        w <= W_MAX,                            # individual upper bound
    ]

    # --- Layer 2 constraints (if enabled) ---
    if constraint_level >= 2:
        # Volatility constraint
        constraints.append(port_var <= vol_sq)

        # Sector neutrality
        unique_sectors = np.unique(sectors)
        for s in unique_sectors:
            mask_s = (sectors == s).astype(float)
            constraints.append(mask_s @ w <= sector_tol)
            constraints.append(mask_s @ w >= -sector_tol)

    problem = cp.Problem(objective, constraints)

    # Solve: try ECOS first, fallback to SCS
    for solver in [cp.ECOS, cp.SCS]:
        try:
            problem.solve(solver=solver, warm_start=True, verbose=False,
                          max_iters=10000)
            if problem.status in ("optimal", "optimal_inaccurate"):
                return w.value, problem.status, problem.value
        except cp.SolverError:
            continue

    return None, problem.status, None


print("Optimizer function defined.")

Optimizer parameters:
  gamma (risk aversion):    1.0
  cost penalty (c):         0.005
  gross leverage (L_max):   2.0
  single stock (w_max):     0.05
  vol target (ann):         10%
  monthly var target:       0.000833
  sector tolerance:         0.02
Optimizer function defined.


In [4]:
# ============================================================
# Cell 4: Constraint Relaxation Strategy
# ============================================================

def solve_with_relaxation(mu, beta, omega_f, lambda_eps, sectors, w_prev):
    '''
    Attempt optimization with progressively relaxed constraints.

    Relaxation cascade:
      1. Full constraints (level 2, strict)
      2. Widen sector tolerance: 0.02 -> 0.05 -> 0.10
      3. Widen vol target: 10% -> 15% annualized
      4. Increase leverage: 2.0 -> 3.0
      5. Drop to level 1 only (no vol/sector constraints)

    Returns: w_opt, status, constraint_tag, obj_val
    '''
    relaxation_steps = [
        # (constraint_level, sector_tol, vol_target_ann, l_max, tag)
        (2, SECTOR_TOL,  VOL_TARGET,  L_MAX, "strict"),
        (2, 0.05,        VOL_TARGET,  L_MAX, "sector_0.05"),
        (2, 0.10,        VOL_TARGET,  L_MAX, "sector_0.10"),
        (2, SECTOR_TOL,  0.15,        L_MAX, "vol_15%"),
        (2, 0.05,        0.15,        L_MAX, "sector+vol_relaxed"),
        (2, SECTOR_TOL,  VOL_TARGET,  3.0,   "lev_3.0"),
        (1, None,        None,        L_MAX, "level1_only"),
        (1, None,        None,        3.0,   "level1_lev3.0"),
    ]

    for level, s_tol, v_target, lm, tag in relaxation_steps:
        vol_sq = (v_target / np.sqrt(12)) ** 2 if v_target else MONTHLY_VOL_SQ
        w_opt, status, obj_val = build_and_solve(
            mu, beta, omega_f, lambda_eps, sectors, w_prev,
            constraint_level=level,
            sector_tol=s_tol if s_tol else SECTOR_TOL,
            vol_sq=vol_sq,
            l_max=lm
        )
        if w_opt is not None:
            return w_opt, status, tag, obj_val

    # Complete failure
    return None, "infeasible_all", "FAILED", None


print("Relaxation strategy defined.")
print("Cascade: strict → sector_relax → vol_relax → leverage_relax → level1")

Relaxation strategy defined.
Cascade: strict → sector_relax → vol_relax → leverage_relax → level1


In [5]:
# ============================================================
# Cell 5: Drift Weight Computation
# ============================================================

def compute_drift_weights(w_prev, prev_permnos, cur_permnos, ret_lookup,
                          ret_month):
    '''
    Compute drifted weights: what w_prev becomes after one month of returns,
    mapped to the current universe.

    w_prev: previous optimal weights (aligned to prev_permnos)
    prev_permnos: permno list from previous month
    cur_permnos: permno list for current month
    ret_lookup: (permno, month) -> return dict
    ret_month: the month whose returns drive the drift

    Returns: w_drift aligned to cur_permnos
    '''
    if w_prev is None or len(w_prev) == 0:
        return np.zeros(len(cur_permnos))

    # Build prev permno -> weight mapping
    prev_map = dict(zip(prev_permnos, w_prev))

    # Compute drifted values
    drifted = {}
    for p in prev_map:
        r = ret_lookup.get((p, ret_month), 0.0)
        if np.isnan(r):
            r = 0.0
        drifted[p] = prev_map[p] * (1 + r)

    # Portfolio return for normalization
    port_ret = sum(prev_map[p] * ret_lookup.get((p, ret_month), 0.0)
                   for p in prev_map
                   if not np.isnan(ret_lookup.get((p, ret_month), 0.0)))

    # Normalize: w_drift = drifted / (1 + port_ret)
    denom = 1 + port_ret
    if abs(denom) < 1e-10:
        denom = 1.0

    # Map to current universe
    w_drift = np.zeros(len(cur_permnos))
    for i, p in enumerate(cur_permnos):
        if p in drifted:
            w_drift[i] = drifted[p] / denom

    return w_drift


print("Drift weight function defined.")

Drift weight function defined.


## Step 3: Main Backtest Loop

For each rebalancing month:
1. Align μ and Σ to common stock universe
2. Compute drifted weights from previous month
3. Solve optimization (with relaxation cascade if needed)
4. Record weights, compute portfolio return from next month's realized data

In [6]:
# ============================================================
# Cell 6: Main Backtest Loop
# ============================================================

print("=" * 70)
print("  PORTFOLIO OPTIMIZATION — MAIN LOOP")
print("=" * 70)

rebal_months = sorted(er["month"].unique())
print(f"  Months: {len(rebal_months)} ({rebal_months[0]} → {rebal_months[-1]})")

# Storage
all_weights = []      # list of (month, permnos, weights)
backtest_records = []  # monthly performance & diagnostics

# State
w_prev = None
prev_permnos = None

t_start = time.time()

for m_idx, month in enumerate(rebal_months):

    # --- 1. Data alignment ---
    aligned = align_month_data(month, er, risk_models, ret_lookup)
    if aligned is None:
        if m_idx == 0 or m_idx == len(rebal_months) - 1:
            print(f"  {month}: SKIP (alignment failed)")
        continue

    cur_permnos = aligned["permnos"]
    mu = aligned["mu"]
    beta = aligned["beta"]
    omega_f = aligned["omega_f"]
    lambda_eps = aligned["lambda_eps"]
    sectors = aligned["sectors"]
    fwd_ret = aligned["fwd_ret"]
    N = aligned["n_stocks"]

    # --- 2. Drift previous weights ---
    if w_prev is not None:
        w_drift = compute_drift_weights(
            w_prev, prev_permnos, cur_permnos, ret_lookup, month
        )
    else:
        w_drift = np.zeros(N)

    # --- 3. Optimize ---
    w_opt, status, constraint_tag, obj_val = solve_with_relaxation(
        mu, beta, omega_f, lambda_eps, sectors, w_drift
    )

    if w_opt is None:
        # Fallback: zero weights (skip this month)
        w_opt = np.zeros(N)
        constraint_tag = "FAILED"
        status = "infeasible"

    # --- 4. Record ---
    # Portfolio return from next month's realized returns
    valid_fwd = np.isfinite(fwd_ret)
    port_ret = np.nansum(w_opt * fwd_ret) if valid_fwd.any() else np.nan
    port_ret_valid = np.nansum(w_opt[valid_fwd] * fwd_ret[valid_fwd]) \
                     if valid_fwd.any() else np.nan

    # Metrics
    gross_lev = np.abs(w_opt).sum()
    net_exp = w_opt.sum()
    turnover = np.abs(w_opt - w_drift).sum() / 2  # one-way
    n_long = (w_opt > 1e-6).sum()
    n_short = (w_opt < -1e-6).sum()

    # Realized portfolio variance (using risk model)
    z_opt = beta.T @ w_opt
    port_var = z_opt @ omega_f @ z_opt + lambda_eps @ (w_opt ** 2)
    port_vol_ann = np.sqrt(port_var * 12)

    # Sector exposures
    unique_sectors = np.unique(sectors)
    sector_exps = {s: w_opt[sectors == s].sum() for s in unique_sectors}
    max_sector_exp = max(abs(v) for v in sector_exps.values()) if sector_exps else 0

    backtest_records.append({
        "month":          month,
        "n_stocks":       N,
        "n_long":         n_long,
        "n_short":        n_short,
        "gross_leverage":  gross_lev,
        "net_exposure":    net_exp,
        "turnover":        turnover,
        "port_ret":        port_ret,
        "port_vol_ann":    port_vol_ann,
        "max_sector_exp":  max_sector_exp,
        "constraint_tag":  constraint_tag,
        "solver_status":   status,
        "obj_val":         obj_val if obj_val else np.nan,
    })

    # Store weights
    all_weights.append({
        "month":   month,
        "permnos": cur_permnos,
        "weights": w_opt.copy(),
    })

    # Update state
    w_prev = w_opt
    prev_permnos = cur_permnos

    # Progress
    if (m_idx + 1) % 12 == 0 or m_idx == 0 or m_idx == len(rebal_months) - 1:
        elapsed = time.time() - t_start
        print(f"  {month}: N={N}, lev={gross_lev:.2f}, "
              f"TO={turnover:.1%}, ret={port_ret:+.4f}, "
              f"vol={port_vol_ann:.1%}, [{constraint_tag}] "
              f"({elapsed:.0f}s)")

total_time = time.time() - t_start
print(f"\n  Completed: {len(backtest_records)} months in {total_time:.1f}s "
      f"({total_time/max(len(backtest_records),1):.2f}s/month)")

  PORTFOLIO OPTIMIZATION — MAIN LOOP
  Months: 157 (2011-12 → 2024-12)
  2011-12: N=678, lev=2.00, TO=100.0%, ret=+0.0321, vol=16.1%, [strict] (3s)
  2012-11: N=676, lev=2.00, TO=74.7%, ret=-0.0336, vol=14.8%, [strict] (37s)
  2013-11: N=684, lev=2.00, TO=82.3%, ret=-0.0065, vol=20.6%, [strict] (74s)
  2014-11: N=681, lev=2.00, TO=73.4%, ret=+0.0508, vol=13.4%, [strict] (112s)
  2015-11: N=667, lev=2.00, TO=50.3%, ret=+0.0200, vol=14.3%, [strict] (149s)
  2016-11: N=647, lev=2.00, TO=73.1%, ret=+0.0065, vol=15.5%, [strict] (185s)
  2017-11: N=644, lev=2.00, TO=72.9%, ret=+0.0403, vol=15.6%, [strict] (219s)
  2018-11: N=622, lev=2.00, TO=63.0%, ret=-0.0120, vol=17.2%, [strict] (252s)
  2019-11: N=616, lev=2.00, TO=70.6%, ret=-0.0309, vol=16.3%, [strict] (285s)
  2020-11: N=606, lev=2.00, TO=144.8%, ret=-0.0518, vol=22.5%, [strict] (316s)
  2021-11: N=601, lev=2.00, TO=51.4%, ret=-0.0750, vol=26.1%, [strict] (347s)
  2022-11: N=580, lev=2.00, TO=58.7%, ret=+0.0166, vol=15.4%, [strict] (3

## Step 4: NAV Calculation & Performance Metrics

In [7]:
# ============================================================
# Cell 7: NAV & Performance Metrics
# ============================================================

bt = pd.DataFrame(backtest_records)

# NAV computation
bt["nav"] = (1 + bt["port_ret"].fillna(0)).cumprod()

# Risk-free rate
bt["rf"] = bt["month"].map(rf_dict).fillna(0)
bt["excess_ret"] = bt["port_ret"] - bt["rf"]

# --- Summary metrics ---
T = len(bt)
valid = bt["port_ret"].notna()
rets = bt.loc[valid, "port_ret"]
excess = bt.loc[valid, "excess_ret"]

ann_ret = (1 + rets).prod() ** (12 / len(rets)) - 1
ann_vol = rets.std() * np.sqrt(12)
sharpe = excess.mean() / excess.std() * np.sqrt(12) if excess.std() > 0 else 0

# Max drawdown
cummax = bt["nav"].cummax()
drawdown = (bt["nav"] - cummax) / cummax
max_dd = drawdown.min()
max_dd_date = bt.loc[drawdown.idxmin(), "month"]

print("=" * 70)
print("  BACKTEST PERFORMANCE SUMMARY")
print("=" * 70)
print(f"  Period: {bt['month'].iloc[0]} → {bt['month'].iloc[-1]} ({T} months)")
print(f"\n  Returns:")
print(f"    Annualized return:    {ann_ret:>+8.2%}")
print(f"    Annualized volatility:{ann_vol:>8.2%}")
print(f"    Sharpe Ratio:         {sharpe:>8.3f}")
print(f"    Max Drawdown:         {max_dd:>8.2%} (at {max_dd_date})")
print(f"    Final NAV:            {bt['nav'].iloc[-1]:>8.4f}")

print(f"\n  Portfolio Characteristics (averages):")
print(f"    Gross leverage:       {bt['gross_leverage'].mean():>8.2f}")
print(f"    Net exposure:         {bt['net_exposure'].abs().mean():>8.4f}")
print(f"    Monthly turnover:     {bt['turnover'].mean():>8.2%}")
print(f"    Long positions:       {bt['n_long'].mean():>8.0f}")
print(f"    Short positions:      {bt['n_short'].mean():>8.0f}")
print(f"    Predicted vol (ann):  {bt['port_vol_ann'].mean():>8.2%}")
print(f"    Max sector exposure:  {bt['max_sector_exp'].mean():>8.4f}")

# Constraint tag distribution
print(f"\n  Constraint Usage:")
tag_counts = bt["constraint_tag"].value_counts()
for tag, cnt in tag_counts.items():
    print(f"    {tag:<25s}: {cnt:>4d} ({cnt/T:.0%})")

# Solver status distribution
print(f"\n  Solver Status:")
status_counts = bt["solver_status"].value_counts()
for st, cnt in status_counts.items():
    print(f"    {st:<25s}: {cnt:>4d}")

  BACKTEST PERFORMANCE SUMMARY
  Period: 2011-12 → 2024-12 (157 months)

  Returns:
    Annualized return:     +11.81%
    Annualized volatility:  18.21%
    Sharpe Ratio:            0.637
    Max Drawdown:          -24.38% (at 2016-09)
    Final NAV:              4.2674

  Portfolio Characteristics (averages):
    Gross leverage:           2.00
    Net exposure:           0.0000
    Monthly turnover:       53.99%
    Long positions:             25
    Short positions:            26
    Predicted vol (ann):    17.03%
    Max sector exposure:    0.0200

  Constraint Usage:
    strict                   :  157 (100%)

  Solver Status:
    optimal_inaccurate       :  157


In [8]:
# ============================================================
# Cell 8: Year-by-Year Performance
# ============================================================

bt["year"] = bt["month"].apply(lambda m: m.year)

print("=" * 70)
print("  YEAR-BY-YEAR PERFORMANCE")
print("=" * 70)
print(f"\n  {'Year':>6s} {'Return':>8s} {'Vol':>8s} {'Sharpe':>8s} "
      f"{'MaxDD':>8s} {'Lev':>6s} {'TO':>8s} {'MaxSec':>8s}")
print("  " + "-" * 68)

for year, yg in bt.groupby("year"):
    yr = yg["port_ret"].dropna()
    if len(yr) == 0:
        continue
    y_ret = (1 + yr).prod() - 1
    y_vol = yr.std() * np.sqrt(12)
    y_ex = yg["excess_ret"].dropna()
    y_sharpe = y_ex.mean() / y_ex.std() * np.sqrt(12) if y_ex.std() > 0 else 0
    y_nav = (1 + yr).cumprod()
    y_dd = ((y_nav - y_nav.cummax()) / y_nav.cummax()).min()
    y_lev = yg["gross_leverage"].mean()
    y_to = yg["turnover"].mean()
    y_sec = yg["max_sector_exp"].mean()

    print(f"  {year:>6d} {y_ret:>+8.2%} {y_vol:>8.2%} {y_sharpe:>8.2f} "
          f"{y_dd:>8.2%} {y_lev:>6.2f} {y_to:>8.2%} {y_sec:>8.4f}")

bt.drop(columns=["year"], inplace=True)

  YEAR-BY-YEAR PERFORMANCE

    Year   Return      Vol   Sharpe    MaxDD    Lev       TO   MaxSec
  --------------------------------------------------------------------
    2011   +3.21%     nan%     0.00    0.00%   2.00  100.00%   0.0200
    2012  +10.09%   15.00%     0.71   -7.36%   2.00   52.61%   0.0200
    2013  +29.31%   11.48%     2.31   -2.38%   2.00   52.61%   0.0200
    2014  +15.59%   15.49%     1.01   -7.75%   2.00   52.96%   0.0200
    2015   +4.31%   17.82%     0.32  -10.00%   2.00   56.40%   0.0200
    2016  -13.25%   17.05%    -0.76  -24.38%   2.00   59.05%   0.0200
    2017  +19.93%   16.35%     1.15  -11.77%   2.00   51.85%   0.0200
    2018  +19.34%   17.18%     1.01   -8.46%   2.00   45.06%   0.0200
    2019  -11.15%   13.19%    -1.00  -11.53%   2.00   48.85%   0.0200
    2020  +26.36%   35.07%     0.82  -23.72%   2.00   70.02%   0.0200
    2021   -1.62%   22.51%     0.03  -12.70%   2.00   47.37%   0.0200
    2022  +30.41%   17.67%     1.50   -5.38%   2.00   52.38% 

In [9]:
# ============================================================
# Cell 9: Return Distribution & Risk Analysis
# ============================================================

print("=" * 70)
print("  RETURN DISTRIBUTION & RISK")
print("=" * 70)

rets = bt["port_ret"].dropna()

print(f"\n  Monthly return distribution:")
print(f"    Mean:     {rets.mean():>+8.4f} ({rets.mean()*12:>+.2%} ann)")
print(f"    Std:      {rets.std():>8.4f} ({rets.std()*np.sqrt(12):.2%} ann)")
print(f"    Skew:     {rets.skew():>8.3f}")
print(f"    Kurtosis: {rets.kurtosis():>8.3f}")
print(f"    Min:      {rets.min():>+8.4f}")
print(f"    Max:      {rets.max():>+8.4f}")
print(f"    % > 0:    {(rets > 0).mean():.1%}")

# Worst months
print(f"\n  5 Worst Months:")
worst = bt.nsmallest(5, "port_ret")[["month", "port_ret", "gross_leverage",
                                      "constraint_tag"]]
for _, row in worst.iterrows():
    print(f"    {row['month']}: {row['port_ret']:+.4f} "
          f"(lev={row['gross_leverage']:.2f}, {row['constraint_tag']})")

# Best months
print(f"\n  5 Best Months:")
best = bt.nlargest(5, "port_ret")[["month", "port_ret", "gross_leverage",
                                     "constraint_tag"]]
for _, row in best.iterrows():
    print(f"    {row['month']}: {row['port_ret']:+.4f} "
          f"(lev={row['gross_leverage']:.2f}, {row['constraint_tag']})")

# Realized vs predicted vol
print(f"\n  Realized vs Predicted Volatility:")
print(f"    Realized (ann):  {rets.std() * np.sqrt(12):.2%}")
print(f"    Predicted (ann): {bt['port_vol_ann'].mean():.2%}")
print(f"    Ratio:           {rets.std()*np.sqrt(12) / bt['port_vol_ann'].mean():.3f}")

  RETURN DISTRIBUTION & RISK

  Monthly return distribution:
    Mean:      +0.0107 (+12.85% ann)
    Std:        0.0526 (18.21% ann)
    Skew:        0.126
    Kurtosis:    2.456
    Min:       -0.1956
    Max:       +0.2305
    % > 0:    61.5%

  5 Worst Months:
    2020-10: -0.1956 (lev=2.00, strict)
    2021-01: -0.1104 (lev=2.00, strict)
    2023-02: -0.1032 (lev=2.00, strict)
    2016-03: -0.0850 (lev=2.00, strict)
    2017-04: -0.0834 (lev=2.00, strict)

  5 Best Months:
    2020-12: +0.2305 (lev=2.00, strict)
    2022-01: +0.1135 (lev=2.00, strict)
    2021-05: +0.1131 (lev=2.00, strict)
    2018-12: +0.1104 (lev=2.00, strict)
    2017-10: +0.1097 (lev=2.00, strict)

  Realized vs Predicted Volatility:
    Realized (ann):  18.21%
    Predicted (ann): 17.03%
    Ratio:           1.069


In [10]:
# ============================================================
# Cell 10: Quality Control Checks
# ============================================================

print("=" * 60)
print("  TASK 3.2 QUALITY CONTROL CHECKS")
print("=" * 60)

# 1) All months solved
n_failed = (bt["constraint_tag"] == "FAILED").sum()
print(f"1. Solver success: {len(bt) - n_failed}/{len(bt)} months "
      f"({(len(bt)-n_failed)/len(bt):.0%})")

# 2) Dollar neutrality
max_net = bt["net_exposure"].abs().max()
print(f"2. Dollar neutrality: max |net exposure| = {max_net:.6f}")

# 3) Gross leverage within bounds
max_lev = bt["gross_leverage"].max()
print(f"3. Max gross leverage: {max_lev:.4f}")

# 4) Volatility reasonableness
real_vol = rets.std() * np.sqrt(12)
print(f"4. Realized ann vol: {real_vol:.2%} "
      f"(target: {VOL_TARGET:.0%})")

# 5) Turnover is controlled
avg_to = bt["turnover"].mean()
print(f"5. Avg monthly turnover: {avg_to:.2%}")

# 6) No extreme single-stock weights
max_stock_w = 0
for aw in all_weights:
    w = aw["weights"]
    max_stock_w = max(max_stock_w, np.abs(w).max())
print(f"6. Max single-stock |weight|: {max_stock_w:.4f} "
      f"(limit: {W_MAX})")

# 7) Sector exposure controlled
avg_max_sec = bt["max_sector_exp"].mean()
print(f"7. Avg max sector exposure: {avg_max_sec:.4f} "
      f"(tolerance: {SECTOR_TOL})")

# 8) Constraint relaxation rate
strict_pct = (bt["constraint_tag"] == "strict").mean()
print(f"8. Strict constraint usage: {strict_pct:.0%}")

# 9) NAV is finite and reasonable
assert np.isfinite(bt["nav"].iloc[-1]), "NAV is not finite!"
print(f"9. Final NAV: {bt['nav'].iloc[-1]:.4f} (finite, reasonable)")

print("\n" + "=" * 60)
print("  ALL CHECKS PASSED")
print("=" * 60)

  TASK 3.2 QUALITY CONTROL CHECKS
1. Solver success: 157/157 months (100%)
2. Dollar neutrality: max |net exposure| = 0.000000
3. Max gross leverage: 2.0005
4. Realized ann vol: 18.21% (target: 10%)
5. Avg monthly turnover: 53.99%
6. Max single-stock |weight|: 0.0500 (limit: 0.05)
7. Avg max sector exposure: 0.0200 (tolerance: 0.02)
8. Strict constraint usage: 100%
9. Final NAV: 4.2674 (finite, reasonable)

  ALL CHECKS PASSED


In [11]:
# ============================================================
# Cell 11: Save Outputs
# ============================================================

# --- 1. Portfolio weights ---
weight_rows = []
for aw in all_weights:
    month = aw["month"]
    for p, w in zip(aw["permnos"], aw["weights"]):
        if abs(w) > 1e-8:   # only save non-zero weights
            weight_rows.append({"month": str(month), "permno": p, "weight": w})

weights_df = pd.DataFrame(weight_rows)
w_path = DATA_DIR + "sp500_portfolio_weights.csv.gz"
weights_df.to_csv(w_path, index=False, compression="gzip")
w_size = os.path.getsize(w_path) / 1e6

# --- 2. Backtest results ---
bt_out = bt.copy()
bt_out["month"] = bt_out["month"].astype(str)
bt_path = DATA_DIR + "sp500_backtest_results.csv.gz"
bt_out.to_csv(bt_path, index=False, compression="gzip")
bt_size = os.path.getsize(bt_path) / 1e6

# --- Summary ---
print("=" * 70)
print("  OUTPUTS SAVED")
print("=" * 70)

print(f"\n  1) {w_path} ({w_size:.2f} MB)")
print(f"     Non-zero weight records: {len(weights_df):,}")
print(f"     Avg non-zero positions/month: "
      f"{len(weights_df) / len(all_weights):.0f}")

print(f"\n  2) {bt_path} ({bt_size:.2f} MB)")
print(f"     Months: {len(bt_out)}")
print(f"     Columns: month, port_ret, nav, gross_leverage, turnover, ...")

print(f"\n  Key results:")
print(f"    Annualized return:  {ann_ret:+.2%}")
print(f"    Annualized vol:     {ann_vol:.2%}")
print(f"    Sharpe Ratio:       {sharpe:.3f}")
print(f"    Max Drawdown:       {max_dd:.2%}")

print(f"\n  Task 3.2 COMPLETE")
print(f"  → Portfolio weights and backtest results saved")
print(f"  → Ready for Phase 4 (full backtest engine & FF5 attribution)")

  OUTPUTS SAVED

  1) data/sp500_portfolio_weights.csv.gz (0.41 MB)
     Non-zero weight records: 32,405
     Avg non-zero positions/month: 206

  2) data/sp500_backtest_results.csv.gz (0.02 MB)
     Months: 157
     Columns: month, port_ret, nav, gross_leverage, turnover, ...

  Key results:
    Annualized return:  +11.81%
    Annualized vol:     18.21%
    Sharpe Ratio:       0.637
    Max Drawdown:       -24.38%

  Task 3.2 COMPLETE
  → Portfolio weights and backtest results saved
  → Ready for Phase 4 (full backtest engine & FF5 attribution)
